# 01 — Data Loading & Cleaning
Sales Performance Analysis & Forecasting project.

This notebook loads the raw Superstore dataset, inspects it, cleans it, and saves a processed version for the rest of the pipeline.

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', 50)
RAW_PATH = '../data/raw/Sample - Superstore.csv'
PROCESSED_PATH = '../data/processed/superstore_clean.csv'
os.makedirs('../data/processed', exist_ok=True)

## 1. Load the raw data
The Kaggle Superstore CSV is usually saved with `latin-1` (or `cp1252`) encoding, not UTF-8 — we handle that here.

In [2]:
def load_raw(path=RAW_PATH):
    for enc in ('utf-8', 'latin-1', 'cp1252'):
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"Loaded with encoding='{enc}' -> shape {df.shape}")
            return df
        except UnicodeDecodeError:
            continue
    raise ValueError("Could not decode file with common encodings")

df = load_raw()
df.head()

Loaded with encoding='utf-8' -> shape (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [4]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Row ID,9994.0,NaN,NaN,NaN,4997.5,2885.163629,1.0,2499.25,4997.5,7495.75,9994.0
Order ID,9994,5009,CA-2017-100111,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order Date,9994,1237,9/5/2016,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Date,9994,1334,12/16/2015,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Mode,9994,4,Standard Class,5968,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer ID,9994,793,WB-21850,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer Name,9994,793,William Brown,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Segment,9994,3,Consumer,5191,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,9994,1,United States,9994,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,9994,531,New York City,915,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Standardize column names
Convert to snake_case so every later notebook can rely on consistent names.

In [5]:
def to_snake(col):
    return (
        col.strip()
           .lower()
           .replace('-', '_')
           .replace(' ', '_')
           .replace('/', '_')
    )

df.columns = [to_snake(c) for c in df.columns]
df.columns.tolist()

['row_id',
 'order_id',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_id',
 'customer_name',
 'segment',
 'country',
 'city',
 'state',
 'postal_code',
 'region',
 'product_id',
 'category',
 'sub_category',
 'product_name',
 'sales',
 'quantity',
 'discount',
 'profit']

## 3. Fix data types
Parse date columns, cast IDs to string (so they are never treated as numeric), and confirm numeric columns are numeric.

In [6]:
date_cols = [c for c in df.columns if 'date' in c]
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors='coerce')

id_cols = [c for c in df.columns if c.endswith('_id')]
for c in id_cols:
    df[c] = df[c].astype(str)

numeric_cols = ['sales', 'quantity', 'discount', 'profit']
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df.dtypes

row_id                   object
order_id                 object
order_date       datetime64[ns]
ship_date        datetime64[ns]
ship_mode                object
customer_id              object
customer_name            object
segment                  object
country                  object
city                     object
state                    object
postal_code               int64
region                   object
product_id               object
category                 object
sub_category             object
product_name             object
sales                   float64
quantity                  int64
discount                float64
profit                  float64
dtype: object

## 4. Missing values

In [7]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

,missing_count,missing_pct


In [8]:
# Rows where a date failed to parse, or a numeric field is missing, are dropped —
# for a sales-order dataset these are usually a small handful of corrupt rows.
before = len(df)
critical_cols = [c for c in date_cols + numeric_cols if c in df.columns]
df = df.dropna(subset=critical_cols)
after = len(df)
print(f"Dropped {before - after} rows with missing critical fields ({(before - after) / before:.2%})")

Dropped 0 rows with missing critical fields (0.00%)


## 5. Duplicates

In [9]:
dupe_count = df.duplicated().sum()
print(f"Exact duplicate rows: {dupe_count}")
df = df.drop_duplicates()
print(f"Shape after dropping duplicates: {df.shape}")

Exact duplicate rows: 0
Shape after dropping duplicates: (9994, 21)


## 6. Outlier check
Flag (but don't blindly drop) extreme sales/profit values — these can be legitimate large orders, which matter for the business story.

In [10]:
for c in ['sales', 'profit', 'discount']:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((df[c] < lower) | (df[c] > upper)).sum()
    print(f"{c}: {n_outliers} potential outliers outside [{lower:.2f}, {upper:.2f}]")

sales: 1167 potential outliers outside [-271.71, 498.93]
profit: 1881 potential outliers outside [-39.72, 70.82]
discount: 856 potential outliers outside [-0.30, 0.50]


In [11]:
# Discount should logically be within [0, 1]; sales/quantity should be positive.
invalid = df[(df['discount'] < 0) | (df['discount'] > 1) | (df['sales'] <= 0) | (df['quantity'] <= 0)]
print(f"Logically invalid rows: {len(invalid)}")
df = df[~df.index.isin(invalid.index)]

Logically invalid rows: 0


## 7. Feature engineering for downstream notebooks

In [12]:
order_date_col = 'order_date' if 'order_date' in df.columns else date_cols[0]

df['order_year'] = df[order_date_col].dt.year
df['order_month'] = df[order_date_col].dt.month
df['order_year_month'] = df[order_date_col].dt.to_period('M').astype(str)
df['order_quarter'] = df[order_date_col].dt.to_period('Q').astype(str)
df['profit_margin'] = np.where(df['sales'] != 0, df['profit'] / df['sales'], np.nan)

if 'ship_date' in df.columns:
    df['fulfillment_days'] = (df['ship_date'] - df[order_date_col]).dt.days

df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,order_year,order_month,order_year_month,order_quarter,profit_margin,fulfillment_days
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,2016,11,2016-11,2016Q4,0.1600,3
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,2016,11,2016-11,2016Q4,0.3000,3
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,2016,6,2016-06,2016Q2,0.4700,4
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,2015,10,2015-10,2015Q4,-0.4000,7
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,2015,10,2015-10,2015Q4,0.1125,7


## 8. Save processed data

In [13]:
df.to_csv(PROCESSED_PATH, index=False)
print(f"Saved cleaned dataset -> {PROCESSED_PATH}")
print(f"Final shape: {df.shape}")

Saved cleaned dataset -> ../data/processed/superstore_clean.csv
Final shape: (9994, 27)


## Summary
- Standardized column names to snake_case
- Parsed dates, cast IDs to string, coerced numeric columns
- Dropped rows with missing critical fields and exact duplicates
- Removed logically invalid rows (negative sales/quantity, discount outside [0,1])
- Added `order_year`, `order_month`, `order_year_month`, `order_quarter`, `profit_margin`, and `fulfillment_days`
- Saved the cleaned file to `data/processed/superstore_clean.csv` for the rest of the pipeline